In [1]:
import torch
from torch.nn import functional as F
from transformers import AutoTokenizer,  AutoModelForSequenceClassification, DataCollatorWithPadding
from dataset import load_dialogue_data
from peft import LoraConfig, TaskType, get_peft_model

# from torch.utils.data import DataLoader
# import os
# import json
# import numpy as np
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# from datasets import load_dataset
# from lion_pytorch import Lion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/root/miniconda3/envs/AiSpea/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the tokenizer for Gemma 2B
# model_id = "unsloth/gemma-2-2b-it" #"google/gemma-2-2b-it"
# model_id = "distilbert-base-uncased"
checkpoint = "Geotrend/distilbert-base-zh-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

id2label = {0: "L1", 1: "L2", 2: "L3"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=3, id2label=id2label, label2id=label2id
)
model.to(device)

# How big is the tokenizer?
print(f"Vocab size: {len(tokenizer.get_vocab())}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at Geotrend/distilbert-base-zh-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Vocab size: 12941


In [89]:
# 2) Transform: tokenize + map “语言能力” → id
def encode(sample):
    text = sample["dialog_item"]["text"]
    toks = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=64,            # adjust if you expect longer utterances
        return_tensors="pt",
    )
    lang_level = sample["labels"].get("语言能力", "L1")  # default to L1 if missing
    toks["labels"] = torch.tensor(label2id[lang_level], dtype=torch.long)
    # squeeze dim 0 so DataLoader can stack tensors
    return {k: v.squeeze(0) for k, v in toks.items()}

In [90]:
train_loader, test_loader = load_dialogue_data(
    "dataset.json",
    test_ratio=0.2,
    batch_size=32,
    shuffle=True,
    transform=encode,
)

collate_fn = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

In [ ]:
def apply_lora(model):
    # Define LoRA config

    peft_cfg = LoraConfig(
    task_type      = TaskType.SEQ_CLS,   # sequence-classification
    r              = 8,                  # rank of LoRA matrices
    lora_alpha     = 16,                 # scaling
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = ["q_lin", "v_lin"], # DistilBERT attention MLPs
    ) # lora_alpha, lora_dropout, bias

    # Apply LoRA to the model
    lora_model = get_peft_model(model, peft_cfg)
    return lora_model

model = apply_lora(model)
print(model.print_trainable_parameters())

trainable params: 740,355 || all params: 54,193,926 || trainable%: 1.3661
None


In [93]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)  # slightly higher LR is fine for LoRA
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for batch in train_loader:
        batch = collate_fn(batch)
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")

        outputs = model(**batch, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}  |  train loss: {running_loss/len(train_loader):.4f}")

    # ------------ quick validation ------------
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            batch = collate_fn(batch)
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            preds  = model(**batch).logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    print(f"val acc: {correct/total:.4%}")

# -------------------------------------------------
# Save LoRA adapter (only ~1 MB)
# -------------------------------------------------
model.save_pretrained("distilbert-ling-lora")

Epoch 1  |  train loss: 1.0101
val acc: 40.0000%
Epoch 2  |  train loss: 0.9904
val acc: 60.0000%
Epoch 3  |  train loss: 0.9830
val acc: 80.0000%
Epoch 4  |  train loss: 0.9681
val acc: 80.0000%
Epoch 5  |  train loss: 0.9576
val acc: 80.0000%
Epoch 6  |  train loss: 0.9380
val acc: 60.0000%
Epoch 7  |  train loss: 0.9102
val acc: 60.0000%
Epoch 8  |  train loss: 0.9030
val acc: 60.0000%
Epoch 9  |  train loss: 0.8742
val acc: 60.0000%
Epoch 10  |  train loss: 0.8546
val acc: 60.0000%


In [101]:
# -------------------------------------------------
# Inference helper
# -------------------------------------------------
def predict_level(sentence: str) -> str:
    model.eval()
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        pred_id = int(model(**inputs).logits.argmax(-1))
    return model.config.id2label[pred_id]

print(predict_level("如果声音跑得慢，雷声就像迟到的快递员"))  # ➜ L3 (example)

L3
